<a href="https://colab.research.google.com/github/victorberriosAP/01_LandingPageContratista/blob/main/Desarrollo_p_tecnica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# ==========================================
# CONFIGURACIÓN DEL ENTORNO (EJECUTAR PRIMERO)
# ==========================================
import pandas as pd
import sqlite3

# 1) URLs de archivos crudos (Raw) en GitHub
urls = {
    'clientes': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/clientes.csv",
    'ventas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/ventas.csv",
    'trabajadores': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/trabajadores.csv",
    'cargos': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/cargos.csv",
    'areas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/areas.csv"
}

print("Preparando base de datos 'techstore.db'.. .")

# 2) Crear la conexión y la base de datos
conn = sqlite3.connect('techstore.db')

# 3) Leer cada CSV e insertarlo como tabla en la base de datos
for tabla, url in urls.items():
    try:
        df = pd.read_csv(url, sep=None, engine='python')
        df.to_sql(tabla, conn, index=False, if_exists='replace')
        print(f"Tabla '{tabla}' cargada exitosamente.")
    except Exception as e:
        print(f"Error al cargar la tabla '{tabla}': {e}")

conn.close()
print("¡Entorno listo! La base de datos relacional está disponible para ser consultada.")

Preparando base de datos 'techstore.db'.. .
Tabla 'clientes' cargada exitosamente.
Tabla 'ventas' cargada exitosamente.
Tabla 'trabajadores' cargada exitosamente.
Tabla 'cargos' cargada exitosamente.
Tabla 'areas' cargada exitosamente.
¡Entorno listo! La base de datos relacional está disponible para ser consultada.


In [ ]:
import sqlite3
import pandas as pd
import re

In [ ]:
#Conectar con base de datos
conn = sqlite3.connect('techstore.db')

##Parte 1: Conexión y Extracción (SQL & Python)

1. Ventas Totales: Escribe una consulta para calcular el ingreso total generado por órdenes con estado "Entregado" y el % del total de ventas que representa.

In [ ]:
df_ventas_totales = pd.read_sql_query(
    """
    SELECT
      SUM(total_dte) AS total_entregado,
      (SUM(total_dte)*100.0) / (SELECT SUM(total_dte)
        FROM ventas) AS porcentaje_entregado
    FROM ventas
    WHERE estado = 'Entregado'
    """,
    conn
)
df_ventas_totales
total = df_ventas_totales.iloc[0, 0]
porcentaje = df_ventas_totales.iloc[0, 1]

#Respuesta escrita
print(f"Se han entregado ${total},"
      f"lo que representa {porcentaje:.2f}% del total vendido."
)


Se han entregado $477191018,lo que representa 81.06% del total vendido.


2. Top Clientes: Encuentra los 5 usuarios que han gastado más dinero históricamente, mostrando su id, rut y el total_dte gastado.

In [ ]:
df_top5_clientes = pd.read_sql_query(
    """
    SELECT
        v.id_cliente,
        c.rut AS rut_cliente,
        SUM(v.total_dte) AS total_gastado
    FROM ventas v
    INNER JOIN clientes c
        ON v.id_cliente = c.id
    GROUP BY v.id_cliente, c.rut
    ORDER BY total_gastado DESC
    LIMIT 5
    """,
    conn
)

#Respuesta
print("Los 5 clientes que han gastado más dinero históricamente son:")
df_top5_clientes


Los 5 clientes que han gastado más dinero históricamente son:


,id_cliente,rut_cliente,total_gastado
0,82,21866669-8,7125123
1,81,10212267-4,5663000
2,94,11686548-3,5621201
3,145,14605016-6,5604037
4,48,19301497-6,5106230


3. Retención: Calcula el porcentaje de usuarios que realizaron una segunda compra dentro de los 200 dte siguientes (si alguien compró el dte 11721 y compró nuevamente antes ó justamente en el dte 11921 es considerado).

In [ ]:
#oObtendremos los clientes en el rango solicitado
df_clientes_retenidos = pd.read_sql_query(
    """
    SELECT COUNT(DISTINCT v1.id_cliente) AS clientes_retenidos
    FROM ventas v1
    JOIN ventas v2
        ON v1.id_cliente = v2.id_cliente
        AND v2.dte > v1.dte
        AND (v2.dte - v1.dte) <= 200
    """,
    conn
)

#Obtenemos el total de clientes únicos
df_total_clientes = pd.read_sql_query(
    """
    SELECT COUNT(DISTINCT id_cliente) AS total_clientes
    FROM ventas
    """,
    conn
)
#Aislamos los valores y obtenemos los resultados
retenidos = df_clientes_retenidos.iloc[0, 0]
total = df_total_clientes.iloc[0, 0]
resultado = (retenidos / total) * 100

#Respuesta escrita
print(
    f"Se han retenido {retenidos} "
    f"de un total de {total} clientes. ")
print(
    f"Esto corresponde a un {resultado:.2f}% de retención de clientes."
)

Se han retenido 173 de un total de 248 clientes. 
Esto corresponde a un 69.76% de retención de clientes.


##Parte 2: Análisis Exploratorio (Python)
Trae las tablas necesarias a DataFrames de Pandas y resuelve:

1. Limpieza: Identifica y muestra los valores nulos o duplicados en la tabla de clientes. Comenta tu criterio.

In [ ]:
# Usando sqlite3 y pandas cargamos la tabla clientes en un DataFrame
df_clientes = pd.read_sql_query(
    """
    SELECT *
    FROM clientes
    """,
    conn
)

# Exploración inicial para revisar contenido
print("- Exploracionde la tabla y contenido")
print(df_clientes.head())
print()

# Revisamos valores nulos
print("- Valores nulos en tabla clientes")
print(df_clientes.isnull().sum())
print()

# Revisamos clientes duplicados
# Se utiliza el RUT como criterio por ser único
duplicados_rut = df_clientes[
    df_clientes.duplicated(subset='rut', keep=False)
]

print()
print("- Clientes duplicados por RUT")
print(duplicados_rut.head())

- Exploracionde la tabla y contenido
   id         rut   nombre segundo_nombre  apellido segundo_apellido  edad  \
0   1  12154051-7   Carlos         Javier    Medina            Rojas  55.0   
1   2  13553384-K  Bárbara       Carolina  Martínez       Valenzuela  29.0   
2   3  17350099-8  Rodrigo      Alejandro   Vergara            Araya  33.0   
3   4  10352896-8  Javiera      Valentina     Silva            Rojas  36.0   
4   5  13841005-6   Matías         Manuel  González            Rojas  61.0   

  fecha_nacimiento  
0       13-04-1971  
1       17-12-1997  
2       02-07-1993  
3       01-05-1990  
4       13-04-1965  

- Valores nulos en tabla clientes
id                  0
rut                 0
nombre              0
segundo_nombre      0
apellido            0
segundo_apellido    0
edad                4
fecha_nacimiento    0
dtype: int64


- Clientes duplicados por RUT
      id         rut   nombre segundo_nombre apellido segundo_apellido  edad  \
3      4  10352896-8  Javiera   

2. Métricas: Identifica el top 10 de total_dte más altos, el top 3 de clientes por volumen de transacciones (cantidad de compras), y el top 3 de vendedores (id_vendedor) con mayor monto acumulado, solo considera vendedores, si hay otros cargos comerciales que han realizado ventas debes ignorarlos.

In [ ]:
#2.1 Identifica el top 10 de total_dte más altos
df_top10_ventas = pd.read_sql_query(
    """
    SELECT
        id,
        total_dte
    FROM ventas
    ORDER BY total_dte DESC
    LIMIT 10
    """,
    conn
)

In [ ]:
#2.2 DataFrame Top's
#Top 3 de clientes por volumen de transacciones (cantidad de compras)
df_top3_clientes = pd.read_sql(
    """
    SELECT
        id_cliente,
        COUNT(*) AS cantidad_compras
    FROM ventas
    GROUP BY id_cliente
    ORDER BY cantidad_compras DESC
    LIMIT 3
    """,
    conn
)

In [ ]:
#top 3 de vendedores con mayor monto acumulado
df_top3_vendedores = pd.read_sql_query(
    """
    SELECT
        v.id_vendedor,
        SUM(v.total_dte) AS monto_acumulado
    FROM ventas v
      -- doble join para encontrar
      -- unir ventas con trabajadores
    INNER JOIN trabajadores t
        ON v.id_vendedor = t.id
        -- unir trabajadores con cargos
    INNER JOIN cargos c
        ON t.id_cargo = c.id
        -- filtrar solo vendedores
    WHERE c.nombre_cargo LIKE '%Ventas%'
    GROUP BY v.id_vendedor
    ORDER BY monto_acumulado DESC
    LIMIT 3
    """,
    conn
)

In [ ]:
#
print("- Top 10 de total_dte más altos")
print(df_top10_ventas)
print()
#
print("- Top 3 Clientes")
print(df_top3_clientes)
print()
#
print("- Top 3 Vendedores")
print(df_top3_vendedores)
print()
#

- Top 10 de total_dte más altos
    id  total_dte
0  321     999825
1  484     998087
2  628     996482
3  662     996396
4  459     996276
5  115     995217
6  735     995031
7  443     994428
8  179     994032
9  208     992339

- Top 3 Clientes
   id_cliente  cantidad_compras
0         147                11
1         148                10
2          84                10

- Top 3 Vendedores
   id_vendedor  monto_acumulado
0            5         40264872
1           13         38625479
2            7         36945022



In [ ]:
#2.3 DataFrame del top 10 de productos mas vendidos
#DataFrame acotado
df_ventas = pd.read_sql_query(
    """
    SELECT id, descripcion
    FROM ventas
    """,
    conn
)

#Con Pandas y re

# separar cada item de la descripcion usando ,
df_ventas["items"] = df_ventas["descripcion"].str.split(",")

# expandir filas (una por producto)
df_exploded = df_ventas.explode("items")

# limpiar espacios
df_exploded["items"] = df_exploded["items"].str.strip()

# extraer cantidad y producto
def parse_item(texto):
                    #r inica que es string d extrae los numeros x es el separador s el espacio
    match = re.match(r"(\d+)x\s(.+)", texto) #r inica que es string d extrae los numeros x es el separador s el espacio
    if match:
                    #divide 2 grupos separasos por la x del texto Grupo 1 cantidad Grupo 2 Texto
        return pd.Series([match.group(2), int(match.group(1))])
    return pd.Series([texto, 1])
              #grupo 2 string                       Grupo 1 int
df_exploded[["producto", "unidades"]] = df_exploded["items"].apply(parse_item)

# top 10 productos
df_top10 = (
    df_exploded
    .groupby("producto", as_index=False)["unidades"]
    .sum()
    .sort_values(by="unidades", ascending=False)
    .head(10)
)

print(df_top10)

                             producto  unidades
2      Escáner de Documentos Portátil       551
9        Mouse Ergonómico Inalámbrico       532
5                Laptop Corporate 14'       504
3                    Hub USB-C 8 en 1       502
7               Memoria RAM 16GB DDR4       492
1              Disco Duro Externo 2TB       492
6   Licencia Software Antivirus 1 Año       486
10     Silla de Escritorio Ergonómica       479
11               Teclado Mecánico RGB       475
0       Auriculares con Micrófono USB       471


##Parte 3: Lógica de Negocio (Hojas de Cálculo / Excel)

In [ ]:
#exportar a excel .xlsx
# nombres de tablas
tablas = ["clientes", "ventas", "trabajadores", "cargos", "areas"]

# crear archivo Excel
with pd.ExcelWriter("techstore.xlsx", engine="openpyxl") as writer:

    for tabla in tablas:
        df = pd.read_sql_query(f"SELECT * FROM {tabla}", conn)
        df.to_excel(writer, sheet_name=tabla, index=False)

conn.close()

print("Archivo techstore.xlsx creado correctamente")

Archivo techstore.xlsx creado correctamente


In [ ]:
#Descargar
from google.colab import files
files.download("techstore.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>